In [1]:
import duckdb
import os
from pathlib import Path

os.chdir(Path.cwd().parent)
con = duckdb.connect(config={'memory_limit': '2GB'})

# Re-create tables (quick version)
con.sql("""
    CREATE OR REPLACE TABLE stg_payments AS 
    SELECT * FROM 'data/silver/date=*/payments_silver.parquet' LIMIT 50000
""")

con.sql("""
    CREATE OR REPLACE TABLE fact_payments AS 
    SELECT 
        ROW_NUMBER() OVER () as payment_key,
        transaction_id,
        amount,
        risk_score,
        status,
        CASE WHEN risk_score > 0 THEN TRUE ELSE FALSE END as is_high_risk,
        DATE(timestamp) as transaction_date
    FROM stg_payments
""")

print("=== FINTECH BUSINESS ANALYTICS ===\n")

con.sql("""
    SELECT 
        COUNT(*) as total_transactions,
        ROUND(SUM(amount), 2) as total_revenue,
        ROUND(AVG(amount), 2) as avg_transaction,
        COUNT(CASE WHEN is_high_risk THEN 1 END) as high_risk_tx,
        ROUND(100.0 * COUNT(CASE WHEN is_high_risk THEN 1 END) / COUNT(*), 2) as high_risk_percentage
    FROM fact_payments
""").show()

print("\nWeek 3 Analytics Complete!")

=== FINTECH BUSINESS ANALYTICS ===

┌────────────────────┬───────────────┬─────────────────┬──────────────┬──────────────────────┐
│ total_transactions │ total_revenue │ avg_transaction │ high_risk_tx │ high_risk_percentage │
│       int64        │    double     │     double      │    int64     │        double        │
├────────────────────┼───────────────┼─────────────────┼──────────────┼──────────────────────┤
│              50000 │    8605359.38 │          172.11 │         3542 │                 7.08 │
└────────────────────┴───────────────┴─────────────────┴──────────────┴──────────────────────┘


Week 3 Analytics Complete!
